# Iris-3B — LoRA pour sprites pixel art 96×96

Ce notebook prépare un premier LoRA à partir des annotations pilotes Qwen3-VL-4B du dataset privé `ethanzxv/pixelart-dataset`. Il récupère seulement les images candidates du pilote, permet de les inspecter, convertit les tags au format demandé, prépare des PNG 96×96, puis lance l’entraînement avec l’intégration Iris-3B d’AI Toolkit.

Le pilote contient 1 000 annotations. La sélection initiale conserve les sprites de personnages, ennemis et créatures, de qualité A/B, sans alerte de taxonomie. Les captions restent des propositions automatiques : inspecte les images et retire les asset IDs incorrects avant l’entraînement.

Iris-3B génère en RGB et ne produit pas de canal alpha. Pour rendre le fond cohérent, les images sont compositées sur blanc. Les sorties seront donc des sprites 96×96 sur fond blanc ; la transparence peut être ajoutée ensuite par détourage, avec vérification visuelle.

Entraînement conseillé sur Linux/Colab avec GPU NVIDIA d’au moins 24 Go. Le modèle de base et son encodeur de texte sont volumineux. Le dataset source est privé : le token HF est demandé en mémoire et n’est jamais écrit dans le notebook.

## 1. Dépendances et GPU

Dans Colab, sélectionne un runtime GPU avant d’exécuter le notebook. Un runtime avec A100 ou L40S 24 Go ou plus est recommandé. En local, Linux avec CUDA est le chemin le plus simple pour AI Toolkit.

In [ ]:
%pip -q install "huggingface_hub>=0.35" pillow matplotlib pyyaml

import json, os, random, re, shutil, subprocess, sys, textwrap
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from getpass import getpass

import torch
from PIL import Image, ImageDraw
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU CUDA détecté. Active un runtime GPU dans Colab ou utilise une machine Linux avec NVIDIA CUDA.")

gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = torch.cuda.is_bf16_supported()
DTYPE = "bf16" if BF16 else "fp16"
print(f"GPU : {gpu_name} — {gpu_gb:.1f} Go — dtype {DTYPE}")
if gpu_gb < 24:
    print("Attention : moins de 24 Go de VRAM. Le chargement ou l’entraînement peut manquer de mémoire.")

WORK_DIR = Path("/content/iris_sprite_lora") if Path("/content").exists() else Path.cwd() / "iris_sprite_lora_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_DATA = WORK_DIR / "dataset_96"
TRAIN_DATA.mkdir(parents=True, exist_ok=True)
print("Dossier de travail :", WORK_DIR.resolve())

## 2. Connexion HF et chargement des annotations

Le jeton doit pouvoir lire le dataset privé. Le fichier d’annotations du pilote est téléchargé depuis le dépôt ; les images seront téléchargées seulement après le filtrage. Utilise une session de notebook de confiance et ne partage pas ses sorties.

In [ ]:
from huggingface_hub import hf_hub_download

DATASET_REPO = "ethanzxv/pixelart-dataset"
ANNOTATION_FILE = "annotations/qwen3vl_fp8_pilot_1000.jsonl"
HF_TOKEN = getpass("Jeton Hugging Face avec accès en lecture au dataset privé : ")

annotation_path = hf_hub_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    filename=ANNOTATION_FILE,
    token=HF_TOKEN,
)
with open(annotation_path, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]
print(f"{len(records)} annotations chargées")

## 3. Sélection des captions candidates

Les champs `train_eligible` du pilote sont tous à false car ils n’ont pas encore été approuvés manuellement et les droits des sources sont inconnus. Le filtre ci-dessous se base sur la catégorie, la qualité et l’absence d’alerte de taxonomie ; il ne transforme pas les labels automatiques en labels humains. Après la planche de revue, retire les images incorrectes dans `REJECT_ASSET_IDS`.

In [ ]:
ALLOWED_CATEGORIES = {"character_sprite", "enemy_sprite", "creature_sprite"}
ALLOWED_QUALITY = {"A_native_clean", "B_usable"}
MAX_TRAIN_IMAGES = None
REJECT_ASSET_IDS = set()  # Ajoute ici les asset_id dont la caption ou l’image est incorrecte.
SEED = 42

candidates = []
for row in records:
    labels = row.get("labels", {})
    if labels.get("asset_category") not in ALLOWED_CATEGORIES:
        continue
    if labels.get("quality_tier") not in ALLOWED_QUALITY:
        continue
    if row.get("needs_taxonomy_review", True):
        continue
    if not row.get("caption_candidate", "").strip():
        continue
    candidates.append(row)

candidates.sort(key=lambda r: r.get("asset_id", ""))
random.Random(SEED).shuffle(candidates)
if MAX_TRAIN_IMAGES is not None:
    candidates = candidates[:MAX_TRAIN_IMAGES]
candidates = [r for r in candidates if r.get("asset_id") not in REJECT_ASSET_IDS]

for term in ["knife", "dagger", "blade", "sword"]:
    count = sum(term in r.get("caption_candidate", "").lower() for r in candidates)
    print(f"Captions mention {term}: {count}")
print(f"Candidats retenus : {len(candidates)}")
print("État des droits dans la sélection :", sorted({r.get("rights_status", "unknown") for r in candidates}))
print("Statut des captions :", sorted({r.get("caption_status", "unknown") for r in candidates}))
if not candidates:
    raise ValueError("Aucun candidat. Vérifie le fichier pilote et les filtres.")

## 4. Télécharger les images retenues

Les images restent dans le cache du runtime et seuls les fichiers retenus sont téléchargés. Cela évite de rapatrier les 100 000 images du dépôt pour ce premier LoRA.

In [ ]:
IMAGE_CACHE = WORK_DIR / "source_images"
IMAGE_CACHE.mkdir(parents=True, exist_ok=True)

def fetch_image(row):
    rel = row["source"]["relative_path"].replace("\\", "/")
    filename = f"data/{rel}"
    local_path = hf_hub_download(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        filename=filename,
        token=HF_TOKEN,
        local_dir=IMAGE_CACHE,
    )
    return row["asset_id"], Path(local_path)

image_paths = {}
errors = []
with ThreadPoolExecutor(max_workers=6) as pool:
    futures = [pool.submit(fetch_image, row) for row in candidates]
    for future in as_completed(futures):
        try:
            asset_id, path = future.result()
            image_paths[asset_id] = path
        except Exception as exc:
            errors.append(str(exc))

if errors:
    print(f"Téléchargements en erreur : {len(errors)}")
    print("Premier message :", errors[0][:500])
print(f"Images locales : {len(image_paths)} / {len(candidates)}")
if not image_paths:
    raise RuntimeError("Aucune image récupérée. Vérifie le token et l’accès au dépôt.")
del HF_TOKEN  # Le jeton n’est plus nécessaire après le téléchargement.

## 5. Revue visuelle rapide

Inspecte la planche et les captions. Le code affiche un échantillon reproductible de 36 images ; change `REVIEW_COUNT` pour en parcourir davantage. Inscris les `asset_id` incorrects dans `REJECT_ASSET_IDS` puis réexécute les cellules de sélection, de téléchargement et de préparation.

In [ ]:
REVIEW_COUNT = min(36, len(image_paths))
review_rows = [r for r in candidates if r["asset_id"] in image_paths][:REVIEW_COUNT]
cols = 6
thumb = 96
cell_w, cell_h = 190, 175
rows_n = (len(review_rows) + cols - 1) // cols
sheet = Image.new("RGB", (cols * cell_w, rows_n * cell_h), "#f2f2f2")
draw = ImageDraw.Draw(sheet)

for i, row in enumerate(review_rows):
    x, y = (i % cols) * cell_w, (i // cols) * cell_h
    im = Image.open(image_paths[row["asset_id"]]).convert("RGBA")
    im.thumbnail((thumb, thumb), Image.Resampling.NEAREST)
    tile = Image.new("RGBA", (thumb, thumb), "white")
    tile.alpha_composite(im, ((thumb - im.width) // 2, (thumb - im.height) // 2))
    sheet.paste(tile.convert("RGB"), (x + (cell_w - thumb) // 2, y + 4))
    draw.text((x + 5, y + 105), row["asset_id"], fill="black")
    caption = row["caption_candidate"].replace("\n", " ")
    draw.multiline_text((x + 5, y + 122), textwrap.fill(caption, width=27)[:105], fill="#333333", spacing=2)
display(sheet)
print("Exemple de caption candidate :", review_rows[0]["caption_candidate"])
print("Pour exclure une ligne, copie son asset_id dans REJECT_ASSET_IDS puis relance les cellules concernées.")

## 6. Conversion au schéma de tags et préparation en 96×96

Les images sont recadrées sur leur zone visible si elles ont de la transparence, redimensionnées en nearest-neighbor, puis centrées dans une toile blanche 96×96. 96 est divisible par 16, la taille de patch de l’intégration Iris-3B. Les captions automatiques qui mentionnent un fond transparent sont corrigées pour correspondre au fond blanc réellement appris.

In [ ]:
import numpy as np

BACKGROUND = (255, 255, 255)
CANVAS_SIZE = 96
SPRITE_MAX_SIZE = 84
DATASET_OUT = TRAIN_DATA
for old in DATASET_OUT.iterdir():
    if old.is_file():
        old.unlink()

def clean_text(value):
    if value is None or value == "":
        return "not_applicable"
    if isinstance(value, list):
        values = [str(v).strip() for v in value if str(v).strip() and str(v).lower() not in {"unknown", "not_applicable"}]
        return values if values else "not_applicable"
    value = str(value).strip()
    return value if value.lower() not in {"unknown", "not_applicable", "none"} else "not_applicable"

def qwen_tags(row):
    labels = row.get("labels", {})
    subject_map = {"character_sprite": "character", "enemy_sprite": "enemy", "creature_sprite": "creature"}
    equipment = clean_text(labels.get("visible_equipment", []))
    return {
        "asset_type": "sprite",
        "subject": subject_map.get(labels.get("asset_category"), clean_text(labels.get("subject_type"))),
        "view": clean_text(labels.get("view")),
        "facing": clean_text(labels.get("facing_direction")),
        "pose": clean_text(labels.get("pose")),
        "colors": clean_text(labels.get("dominant_color", [])),
        "hair": "not_applicable",
        "clothing": "not_applicable",
        "accessories": "not_applicable",
        "equipment": equipment,
        "background": "plain white",
        "rendering_style": "pixel art",
    }

def caption_for_white_background(text):
    text = re.sub(r"(?:against|on) a transparent background", "on a plain white background", text, flags=re.I)
    text = re.sub(r"transparent background", "plain white background", text, flags=re.I)
    text = re.sub(r"\s+", " ", text).strip()
    if "background" not in text.lower():
        text = text.rstrip(" .") + ", on a plain white background."
    return text

def prepare_image(source_path, output_path):
    im = Image.open(source_path).convert("RGBA")
    alpha = im.getchannel("A")
    bbox = alpha.getbbox()
    if bbox:
        im = im.crop(bbox)
    w, h = im.size
    scale = min(SPRITE_MAX_SIZE / w, SPRITE_MAX_SIZE / h)
    new_size = (max(1, round(w * scale)), max(1, round(h * scale)))
    im = im.resize(new_size, Image.Resampling.NEAREST)
    canvas = Image.new("RGBA", (CANVAS_SIZE, CANVAS_SIZE), (*BACKGROUND, 255))
    canvas.alpha_composite(im, ((CANVAS_SIZE - im.width) // 2, (CANVAS_SIZE - im.height) // 2))
    canvas.convert("RGB").save(output_path, format="PNG", optimize=True)

training_rows = []
for row in candidates:
    asset_id = row["asset_id"]
    if asset_id not in image_paths or asset_id in REJECT_ASSET_IDS:
        continue
    image_name = f"{asset_id}.png"
    out_image = DATASET_OUT / image_name
    prepare_image(image_paths[asset_id], out_image)
    free_caption = caption_for_white_background(row["caption_candidate"])
    tags = qwen_tags(row)
    training_caption = f"sprite, pixel art, 96x96\n\n{free_caption}"
    (DATASET_OUT / f"{asset_id}.txt").write_text(training_caption, encoding="utf-8")
    training_rows.append({
        "asset_id": asset_id,
        "source": row["source"],
        "tags": tags,
        "free_caption": free_caption,
        "training_caption": training_caption,
        "image": image_name,
        "caption_status": row.get("caption_status"),
        "review_status": row.get("review_status"),
        "rights_status": row.get("rights_status", "unknown"),
    })

with open(WORK_DIR / "qwen3vl_sprite_training.jsonl", "w", encoding="utf-8") as f:
    for row in training_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Prêt : {len(training_rows)} paires image-caption dans {DATASET_OUT}")
print(json.dumps(training_rows[0], ensure_ascii=False, indent=2)[:1800])

In [ ]:
sample_path = DATASET_OUT / training_rows[0]["image"]
sample = Image.open(sample_path)
print("Dimensions :", sample.size, "mode :", sample.mode)
display(sample.resize((384, 384), Image.Resampling.NEAREST))
print((DATASET_OUT / f"{training_rows[0]['asset_id']}.txt").read_text(encoding="utf-8"))

## 7. Installer AI Toolkit

L’intégration Iris-3B est fournie dans AI Toolkit. Le dépôt publie des configurations Iris avec `arch: iris3b`, `quantize_te: true`, `timestep_type: shift` et gradient checkpointing. Le clonage ci-dessous suit la branche principale pour récupérer cette intégration.

In [ ]:
AI_TOOLKIT_DIR = WORK_DIR / "ai-toolkit"
if not AI_TOOLKIT_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/ostris/ai-toolkit.git", str(AI_TOOLKIT_DIR)], check=True)
subprocess.run(["git", "-C", str(AI_TOOLKIT_DIR), "submodule", "update", "--init", "--recursive"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(AI_TOOLKIT_DIR / "requirements.txt")], check=True)
print("AI Toolkit prêt :", AI_TOOLKIT_DIR)

## 8. Configuration et lancement du LoRA

Le premier run utilise rank 16, batch 1 et 2 000 steps. Le DiT et l’encodeur texte sont quantifiés en FP8 via AI Toolkit, avec low-VRAM et cache des embeddings texte ; les calculs d’entraînement restent en BF16/FP16. Les images et previews sont à 96×96. Cela peut réduire la VRAM, mais ne garantit pas que 8 Go suffiront : le pixel head reste en précision supérieure et les activations/échantillons prennent aussi de la mémoire.

In [ ]:
import yaml

RUN_NAME = "iris_pixelart_sprite_96"
OUTPUT_DIR = WORK_DIR / "output"
CONFIG_PATH = WORK_DIR / f"{RUN_NAME}.yaml"

config = {
    "job": "extension",
    "config": {
        "name": RUN_NAME,
        "process": [{
            "type": "sd_trainer",
            "training_folder": str(OUTPUT_DIR),
            "device": "cuda:0",
            "network": {"type": "lora", "linear": 16, "linear_alpha": 16},
            "save": {"dtype": "float16", "save_every": 500, "max_step_saves_to_keep": 3, "push_to_hub": False},
            "datasets": [{
                "folder_path": str(DATASET_OUT),
                "caption_ext": "txt",
                "caption_dropout_rate": 0.02,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "resolution": [96],
            }],
            "train": {
                "batch_size": 1,
                "steps": 2000,
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "cache_text_embeddings": True,
                "noise_scheduler": "flowmatch",
                "timestep_type": "shift",
                "optimizer": "adamw8bit",
                "lr": 0.00005,
                "dtype": DTYPE,
            },
            "model": {
                "arch": "iris3b",
                "name_or_path": "speridlabs/iris-3b",
                "quantize": True,
                "qtype": "qfloat8",
                "quantize_te": True,
                "qtype_te": "qfloat8",
                "low_vram": True,
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": 250,
                "width": 96,
                "height": 96,
                "prompts": [
                    "sprite, pixel art, 96x96\n\na character standing with a knife",
                    "sprite, pixel art, 96x96\n\na small green creature facing front",
                    "sprite, pixel art, 96x96\n\nan enemy character holding a sword",
                ],
                "neg": "",
                "seed": 42,
                "walk_seed": False,
                "guidance_scale": 3.0,
                "sample_steps": 30,
            },
        }],
    },
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(CONFIG_PATH.read_text(encoding="utf-8"))

In [ ]:
RUN_TRAINING = True  # Passe à False si tu veux seulement préparer les fichiers et la config.
if RUN_TRAINING:
    subprocess.run([sys.executable, "run.py", str(CONFIG_PATH)], cwd=AI_TOOLKIT_DIR, check=True)
else:
    print("Config prête. Pour lancer plus tard : python run.py", CONFIG_PATH)

## 9. Récupérer le LoRA et examiner les previews

Les adapters et images d’échantillonnage sont enregistrés sous `output/iris_pixelart_sprite_96`. Vérifie plusieurs checkpoints : le dernier n’est pas toujours le meilleur. Les poids sont un LoRA AI Toolkit pour Iris-3B ; conserve le modèle de base `speridlabs/iris-3b` à l’inférence.

In [ ]:
lora_files = sorted(OUTPUT_DIR.rglob("*.safetensors")) if OUTPUT_DIR.exists() else []
print(f"Fichiers safetensors trouvés : {len(lora_files)}")
for path in lora_files:
    print(path, f"({path.stat().st_size / 1024**2:.1f} MiB)")
print("Dossier de sortie :", OUTPUT_DIR.resolve())

## Notes

- Le pilote contient des captions `candidate_unreviewed`. La revue visuelle et les corrections des captions sont importantes avant d’augmenter le nombre de steps.
- Les annotations pilotes signalent des droits `unknown`. Vérifie les licences des sources avant de publier ou redistribuer un LoRA entraîné dessus.
- Les exemples originaux ont des tailles variées ; le notebook recadre les pixels visibles, les redimensionne en nearest-neighbor et les place sur une toile 96×96 blanche.
- Iris-3B est un modèle RGB sans alpha. Pour produire de la transparence, il faut détourer les sorties après génération et inspecter les contours.
- Le pilote de 1 000 images est un point de départ. Le filtre actuel retient environ 500 exemples ; pour un modèle plus général, il faudra captionner et vérifier davantage de sprites du dataset complet.
- Dans ces captions filtrées, aucun exemple ne mentionne un couteau (1 dague, 2 lames, 289 épées au dernier contrôle). Ajoute des exemples de couteaux captionnés si cette caractéristique doit être fiable.